# FlashSpec Colab + Google Drive 运行脚本

请先在 Colab 菜单选择 `Runtime -> Change runtime type -> GPU`；如需验证 A100，硬件加速器选 A100。

这个 notebook 会把代码仓库放在 Google Drive：`/content/drive/MyDrive/FlashSpecColab`，benchmark JSON 结果放在：`/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/`。这样 Colab runtime 重启后，代码和结果依然保留在云端。

## 0. 挂载 Google Drive

第一次运行会弹出授权。授权后，Colab 可以读写你的 Google Drive。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# 代码直接放在 FlashSpecColab 下（不再套一层 repo）。
# results 是 repo 自带的结果目录，clone 后再创建 colab_kernels 子目录。
PROJECT_DIR = Path("/content/drive/MyDrive/FlashSpecColab")
RESULTS_DIR = PROJECT_DIR / "results" / "colab_kernels"

print("PROJECT_DIR:", PROJECT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## 1. 检查 Colab GPU 环境

如果这里报错，说明当前 runtime 不是 GPU，需要先切换 Colab runtime。

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("当前 Colab runtime 没有 CUDA GPU，请先切换到 GPU/A100 runtime。")

## 2. 从 Google Drive 检出代码并安装依赖

如果仓库已经在 `/content/drive/MyDrive/FlashSpecColab/.git`，则执行 `git pull --ff-only`，否则 clone 到该目录。`.[triton]` 会安装 Triton 可选依赖，用于跑真实 Kernel 1。

In [ ]:
import os
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/FlashSpecColab")
RESULTS_DIR = PROJECT_DIR / "results" / "colab_kernels"

# FlashSpecColab 本身就是 git 仓库检出点（results/ 随 repo 一起来）。
if (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/honey-floria/FlashSpec.git", str(PROJECT_DIR)],
        check=True,
    )
else:
    raise RuntimeError(
        f"{PROJECT_DIR} 已存在但不是 Git 仓库。请先把它变成 FlashSpec 的检出点"
        "（git init + remote，或删空后重新 clone）。"
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print("cwd:", Path.cwd())

%pip install -e ".[triton]"

## 3. 检查 FlashSpec、CUDA 和 Triton 状态

`HAS_TRITON=True` 且 `cuda available=True` 时，`triton_fused` 和 `triton_paged` benchmark 会走真实 Triton kernel。

In [ ]:
import sys
import torch
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/FlashSpecColab")
sys.path.insert(0, str(PROJECT_DIR / "src"))

from flashspec.triton_kernels import HAS_TRITON

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("HAS_TRITON:", HAS_TRITON)

## 3.5 定位 / 安装 Nsight Compute (ncu)

`ncu` 用于采集实测 DRAM 带宽、占用率、SM 利用率，回填 `measured_*` 字段（`--profile-ncu`）。Colab 的 CUDA 安装通常自带 ncu；找不到会尝试 apt 安装。即使没有 ncu，benchmark 仍会产出估算带宽，只是 `measured_*` 为空。

In [ ]:
import os
import shutil
import subprocess

# 优先用 PATH 里的 ncu；否则找 CUDA 默认安装路径；再否则尝试 apt 安装。
NCU_BIN = shutil.which("ncu")
if NCU_BIN is None:
    for cand in ["/usr/local/cuda/bin/ncu", "/opt/nvidia/nsight-compute/ncu"]:
        if os.path.exists(cand):
            NCU_BIN = cand
            break
if NCU_BIN is None:
    print("未找到 ncu，尝试 apt 安装 nsight-compute ...")
    subprocess.run(["apt-get", "-qq", "install", "-y", "nsight-compute"], check=False)
    NCU_BIN = shutil.which("ncu") or "/usr/local/cuda/bin/ncu"

# 把 ncu 所在目录加进 PATH，方便后续 shell cell 直接调用。
if os.path.exists(NCU_BIN):
    os.environ["PATH"] = os.path.dirname(NCU_BIN) + os.pathsep + os.environ.get("PATH", "")
    ver = subprocess.run([NCU_BIN, "--version"], capture_output=True, text=True)
    print("NCU_BIN:", NCU_BIN)
    print(ver.stdout.splitlines()[0] if ver.stdout else ver.stderr[:200])
    HAS_NCU = True
else:
    print("仍未找到 ncu；--profile-ncu 会跳过实测字段（只保留估算带宽）。")
    HAS_NCU = False

# 供后续 cell 通过 {NCU_BIN} 插值使用。
os.environ["NCU_BIN"] = NCU_BIN if os.path.exists(NCU_BIN) else "ncu"

## 4. 运行正确性测试

这里会包含 CUDA + Triton 可用时的 Kernel 1 / Kernel 2 correctness 测试：输出对齐 PyTorch reference，并检查 `materializes_dense_kv == 0.0`。

In [ ]:
%cd /content/drive/MyDrive/FlashSpecColab
!python -m unittest discover -s tests

## 5. A100 目标 shape benchmark

覆盖 `head_dim=64/128` × `seq_len=512/2048/4096`，跑 Kernel 1 Triton fused 与 Kernel 2 Triton paged 主路径，结果 JSON 保存到 Google Drive 的 `/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/`。如果 Colab 时间紧张，可自行调小 `--batch` 或 `--iters`。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "/content/drive/MyDrive/FlashSpecColab"
OUT = f"{REPO}/results/colab_kernels"
os.chdir(REPO)
Path(OUT).mkdir(parents=True, exist_ok=True)
NCU_BIN = os.environ.get("NCU_BIN", "ncu")

# 目标 shape：head_dim=64/128 × seq_len=512/2048/4096，覆盖 Triton 主路径。
SHAPES = [(s, d) for d in (64, 128) for s in (512, 2048, 4096)]
BACKENDS = ["triton_fused", "triton_paged"]

for backend in BACKENDS:
    for seq, hd in SHAPES:
        out = f"{OUT}/{backend}_s{seq}_d{hd}.json"
        cmd = [
            "python", "benchmarks/microbench.py", "--backend", backend,
            "--batch", "16", "--heads", "32", "--seq-len", str(seq),
            "--head-dim", str(hd), "--block-size", "16",
            "--iters", "50", "--warmup", "10", "--repeats", "20",
            "--device", "cuda", "--dtype", "float16", "--json", "--include-raw",
            # --profile-ncu：用 ncu 采集实测带宽/占用率并回填 measured_*（按 kernel 名过滤）。
            "--profile-ncu", "--ncu-bin", NCU_BIN,
            "--output", out,
        ]
        print(">>", backend, f"s{seq} d{hd}")
        subprocess.run(cmd, check=True)

## 6. 汇总所有结果

读取 Google Drive 上的 `/content/drive/MyDrive/FlashSpecColab/results/colab_kernels/*.json`，快速查看每个 benchmark 的关键字段。

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("/content/drive/MyDrive/FlashSpecColab/results/colab_kernels")

def g(d, k, nd=4):
    v = d.get(k)
    return round(float(v), nd) if isinstance(v, (int, float)) else v

for path in sorted(RESULTS_DIR.glob("*.json")):
    d = json.loads(path.read_text())
    est = g(d, "estimated_effective_quant_kv_bandwidth_gbps", 1)
    meas = g(d, "measured_achieved_bandwidth_gbps", 1)
    print(
        path.name,
        "| backend=", d.get("backend"),
        "| splits=", d.get("num_splits"),
        "| block_n=", d.get("block_n"),
        "| lat_ms=", g(d, "latency_ms"),
        "| tok/s=", g(d, "tokens_per_second", 1),
        "| est_bw=", est,
        "| meas_bw=", meas,
        "| occ%=", g(d, "measured_occupancy_pct", 1),
        "| sm%=", g(d, "measured_sm_utilization_pct", 1),
        "| kernels=", d.get("measured_ncu_kernel_names"),
    )
    if d.get("profiler_warning"):
        print("   [WARN]", d["profiler_warning"])
    if d.get("profiler_error"):
        print("   [ERR ]", d["profiler_error"])

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

RESULTS_DIR = Path("/content/drive/MyDrive/FlashSpecColab/results/colab_kernels")

rows = []
for path in sorted(RESULTS_DIR.glob("*.json")):
    d = json.loads(path.read_text())
    occ = d.get("measured_occupancy_pct")
    if not isinstance(occ, (int, float)):
        continue
    seq = d.get("seq_len") or d.get("sequence_length") or "?"
    hd = d.get("head_dim") or "?"
    if path.name.startswith("ab_"):
        mode = "OFF" if "splitoff" in path.stem else "ON"
        label = f"s{seq}\nSplit-K {mode}"
    else:
        label = f"{d.get('backend', path.stem)}\ns{seq}/d{hd}"
    rows.append((label, float(occ)))

if rows:
    labels, occs = zip(*rows)
    fig_w = max(10, 0.75 * len(rows))
    fig, ax = plt.subplots(figsize=(fig_w, 4.8))
    bars = ax.bar(range(len(rows)), occs, color="#4C78A8")
    ax.set_ylabel("Measured occupancy (%)")
    ax.set_title("Occupancy by benchmark method")
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylim(0, max(100, max(occs) * 1.15))
    ax.grid(axis="y", alpha=0.25)
    for bar, occ in zip(bars, occs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{occ:.1f}%", ha="center", va="bottom", fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("没有可绘制的 measured_occupancy_pct；请先用 --profile-ncu 生成实测占用率。")

## 7. 生成对比图和真实带宽表

用 `analyze_results.py` 汇总所有结果 JSON，生成对比柱状图、Triton scaling 图和 `summary.csv`。有 `--profile-ncu` 的实测带宽时，scaling 图会自动叠加实测值（图例标 `[measured]`）。

In [ ]:
import os
from pathlib import Path
from IPython.display import Image, display

REPO = "/content/drive/MyDrive/FlashSpecColab"
RESULTS_DIR = f"{REPO}/results/colab_kernels"
ANALYSIS_DIR = f"{RESULTS_DIR}/analysis"
os.chdir(REPO)

!python scripts/analyze_results.py --results-dir "{RESULTS_DIR}" --output-dir "{ANALYSIS_DIR}"

for png in ["backend_comparison_s2048_d128.png", "triton_scaling.png"]:
    p = Path(ANALYSIS_DIR) / png
    if p.exists():
        display(Image(filename=str(p)))

## 8. Split-K A/B 测试（隔离 Split-K 的纯贡献）

`num_warps` 已固定为 4，这里只切换 Split-K 开/关：

- **OFF**：`FLASHSPEC_NUM_SPLITS=1` 强制走单 kernel 快路径；
- **ON**：不设该环境变量，走自适应 S（s2048→4，s4096→8）。

这样唯一变量就是 Split-K，延迟差就是 Split-K 的纯贡献。第一个 cell 跑 4 组 benchmark，第二个出对比表（含占用率、寄存器、理论占用率，用于判定瓶颈是否受寄存器限制）。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "/content/drive/MyDrive/FlashSpecColab"
OUT = f"{REPO}/results/colab_kernels"
os.chdir(REPO)
Path(OUT).mkdir(parents=True, exist_ok=True)
NCU_BIN = os.environ.get("NCU_BIN", "ncu")

def run_ab(seq, num_splits_override):
    # num_splits_override=None -> ON(自适应)；="1" -> OFF(强制单 kernel)
    tag = "splitoff" if num_splits_override == "1" else "spliton"
    out = f"{OUT}/ab_s{seq}_{tag}.json"
    env = dict(os.environ)
    if num_splits_override is not None:
        env["FLASHSPEC_NUM_SPLITS"] = num_splits_override
    else:
        env.pop("FLASHSPEC_NUM_SPLITS", None)
    cmd = [
        "python", "benchmarks/microbench.py", "--backend", "triton_fused",
        "--batch", "16", "--heads", "32", "--seq-len", str(seq),
        "--head-dim", "128", "--block-size", "16",
        "--iters", "50", "--warmup", "10", "--repeats", "20",
        "--device", "cuda", "--dtype", "float16", "--json",
        # --profile-ncu：采集 occupancy/带宽 + 寄存器/理论占用率，验证瓶颈到底是什么。
        # FLASHSPEC_NUM_SPLITS 通过 env 传给内层 microbench 及 ncu 子进程。
        "--profile-ncu", "--ncu-bin", NCU_BIN,
        "--output", out,
    ]
    print(f">> s{seq} {tag}")
    subprocess.run(cmd, check=True, env=env)

# s2048 与 s4096，各跑 OFF 与 ON
for seq in (2048, 4096):
    run_ab(seq, "1")    # OFF
    run_ab(seq, None)   # ON
print("done")

In [ ]:
import json
from pathlib import Path

OUT = Path("/content/drive/MyDrive/FlashSpecColab/results/colab_kernels")

def lat(d):
    # 优先用 ncu 测得的 kernel 时间；没有则回退到 CUDA event 端到端延迟。
    return d.get("measured_kernel_latency_ms") or d["latency_ms"]

def num(d, k):
    x = d.get(k)
    return f"{x:.1f}" if isinstance(x, (int, float)) else "-"

print(f"{'seq':>6} {'OFF ms':>8} {'ON ms':>8} {'splits':>6} {'speed':>6} "
      f"{'occOFF':>7} {'occON':>7} {'reg':>5} {'theoOcc':>8}  verdict")
for seq in (2048, 4096):
    off = json.loads((OUT / f"ab_s{seq}_splitoff.json").read_text())
    on = json.loads((OUT / f"ab_s{seq}_spliton.json").read_text())
    lo, ln = lat(off), lat(on)
    sp = lo / ln
    v = "Split-K 更快" if sp > 1.03 else ("持平" if sp > 0.97 else "Split-K 更慢")
    # reg/theoOcc 取 OFF（单 kernel）：若理论占用率 ≈ 实测占用率 => 受资源(寄存器)限制，
    # 加 block 无用，需降寄存器；若理论占用率远高于实测 => 是别的调度问题。
    print(f"{seq:>6} {lo:>8.4f} {ln:>8.4f} {str(on.get('num_splits')):>6} {sp:>5.2f}x "
          f"{num(off,'measured_occupancy_pct'):>7} {num(on,'measured_occupancy_pct'):>7} "
          f"{num(off,'measured_registers_per_thread'):>5} "
          f"{num(off,'measured_theoretical_occupancy_pct'):>8}  {v}")

    for tag, d in (("OFF", off), ("ON", on)):
        if d.get("profiler_warning"):
            print(f"    [{tag} WARN] {d['profiler_warning']}")
        if d.get("profiler_error"):
            print(f"    [{tag} ERR ] {d['profiler_error']}")

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

OUT = Path("/content/drive/MyDrive/FlashSpecColab/results/colab_kernels")

seqs = (2048, 4096)
labels = [str(seq) for seq in seqs]
off_occs = []
on_occs = []
for seq in seqs:
    off = json.loads((OUT / f"ab_s{seq}_splitoff.json").read_text())
    on = json.loads((OUT / f"ab_s{seq}_spliton.json").read_text())
    off_occs.append(off.get("measured_occupancy_pct"))
    on_occs.append(on.get("measured_occupancy_pct"))

if all(isinstance(x, (int, float)) for x in off_occs + on_occs):
    x = list(range(len(seqs)))
    width = 0.36
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    off_bars = ax.bar([i - width / 2 for i in x], off_occs, width, label="Split-K OFF", color="#F58518")
    on_bars = ax.bar([i + width / 2 for i in x], on_occs, width, label="Split-K ON", color="#54A24B")
    ax.set_ylabel("Measured occupancy (%)")
    ax.set_title("Split-K occupancy comparison")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("seq_len")
    ax.set_ylim(0, max(100, max(off_occs + on_occs) * 1.15))
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    for bars in (off_bars, on_bars):
        for bar in bars:
            occ = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, occ, f"{occ:.1f}%", ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    plt.show()
else:
    print("Split-K A/B JSON 中没有完整 measured_occupancy_pct；请先用 --profile-ncu 重新生成。")

## 9. 实验 4：降寄存器（block_n A/B）

实验 3 确认占用率天花板被寄存器压死：单 kernel 96 reg/thread → 每 SM 只能放 5 个 block → 理论占用率 31.25%。

`k_deq`/`v_deq` 临时 tile 是 `[block_n, block_d]`，寄存器占用与 `block_n` 成正比。这里固定 `FLASHSPEC_NUM_SPLITS=1`（单 kernel 路径，和实验 3 基线对齐），只变 `block_n`：

- **block_n=64**：默认（基线，应还原出 reg≈96 / theoOcc≈31.25%）；
- **block_n=32**：临时 tile 减半，可能降低 reg、抬高理论占用率，但循环轮数和地址/控制开销会增加。

关键看两步：先看 `reg/theoOcc` 是否改善，再看 `dram%/bw/lat` 是否同步改善。若 reg 降了但带宽和延迟变差，说明缩小 tile 的额外开销抵消了 occupancy 收益，不能作为默认优化。

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "/content/drive/MyDrive/FlashSpecColab"
OUT = f"{REPO}/results/colab_kernels"
os.chdir(REPO)
Path(OUT).mkdir(parents=True, exist_ok=True)
NCU_BIN = os.environ.get("NCU_BIN", "ncu")

def run_blockn(seq, block_n):
    out = f"{OUT}/bn_s{seq}_bn{block_n}.json"
    env = dict(os.environ)
    env["FLASHSPEC_NUM_SPLITS"] = "1"      # 固定单 kernel 路径，隔离 block_n 变量
    env["FLASHSPEC_BLOCK_N"] = str(block_n)
    cmd = [
        "python", "benchmarks/microbench.py", "--backend", "triton_fused",
        "--batch", "16", "--heads", "32", "--seq-len", str(seq),
        "--head-dim", "128", "--block-size", "16",
        "--iters", "50", "--warmup", "10", "--repeats", "20",
        "--device", "cuda", "--dtype", "float16", "--json",
        "--profile-ncu", "--ncu-bin", NCU_BIN,
        "--output", out,
    ]
    print(f">> s{seq} block_n={block_n}")
    subprocess.run(cmd, check=True, env=env)

for seq in (2048, 4096):
    for bn in (64, 32):
        run_blockn(seq, bn)
print("done")

In [ ]:
import json
from pathlib import Path

OUT = Path("/content/drive/MyDrive/FlashSpecColab/results/colab_kernels")

def lat(d):
    return d.get("measured_kernel_latency_ms") or d["latency_ms"]

def num(d, k):
    x = d.get(k)
    return f"{x:.1f}" if isinstance(x, (int, float)) else "-"

print(f"{'seq':>6} {'bn':>4} {'json_bn':>7} {'lat ms':>8} {'reg':>5} {'theoOcc':>8} {'occ':>6} "
      f"{'dram%':>6} {'bw':>6}")
for seq in (2048, 4096):
    base = None
    for bn in (64, 32):
        d = json.loads((OUT / f"bn_s{seq}_bn{bn}.json").read_text())
        l = lat(d)
        # 相对 block_n=64 基线的加速比；小于 1 表示更慢。
        if bn == 64:
            base = l
            sp = ""
        else:
            sp = f"  ({base / l:.2f}x vs bn64)"
        print(f"{seq:>6} {bn:>4} {str(d.get('block_n')):>7} {l:>8.4f} {num(d,'measured_registers_per_thread'):>5} "
              f"{num(d,'measured_theoretical_occupancy_pct'):>8} {num(d,'measured_occupancy_pct'):>6} "
              f"{num(d,'measured_dram_throughput_pct'):>6} {num(d,'measured_achieved_bandwidth_gbps'):>6}{sp}")
        if d.get("profiler_warning"):
            print(f"    [WARN] {d['profiler_warning']}")
    print()
print("判读：先看 block_n 是否写入 JSON，再看 reg/theoOcc 是否改善；")
print("     如果 reg 降了但 dram%/bw/lat 变差，说明缩小 tile 不是净收益。")